# Detekcja odpadów — baseline YOLO11n

Notebook dokumentuje walidację istniejącego datasetu, EDA dla detekcji obiektów, trening pretrained YOLO Nano, GreenAI i końcową ewaluację. Oryginalnych anotacji i podziału nie modyfikujemy.

## 1. Konfiguracja i struktura danych

`data.yaml` definiuje sześć klas oraz istniejące katalogi `train`, `valid`, `test`. Część adnotacji to 5-wartościowe bounding boxy, a część to poligony YOLO. Ultralytics wyznacza ramki z rzeczywistych poligonów przy treningu detekcji.

In [ ]:
from pathlib import Path
import json, pandas as pd, yaml
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
DATA = ROOT / 'Trash Detection Dataset' / 'CUSTOM_DATASET'
RUN = ROOT / 'runs' / 'baseline_yolo_nano'
print(yaml.safe_load((DATA / 'data.yaml').read_text()))

## 2. Audyt jakości danych

Komenda skanuje każdy obraz i każdy wiersz adnotacji, zapisuje pełną listę problemów oraz grupy identycznych obrazów. Dane pozostają bez zmian.

In [ ]:
# Uruchom przed treningiem lub po zmianie datasetu:
# !python -m ml_baseline.audit --dataset 'Trash Detection Dataset/CUSTOM_DATASET' --out runs/academic_3_epoch_example
summary = json.loads((RUN / 'dataset_summary.json').read_text())
issues = json.loads((RUN / 'audit_issues.json').read_text())
display(pd.DataFrame(summary['splits']).T)
print('Klasy:', summary['classes'])
print('Typy anotacji:', summary['annotation_types'])
print('Problemy:', summary['issue_counts'])
display(pd.DataFrame(issues).head(20))

## 3. EDA — klasy i istniejący split

Zliczenia dotyczą obiektów, nie tylko zdjęć. Porównanie splitów pomaga ocenić balans i przesunięcie rozkładów.

In [ ]:
obj = pd.read_csv(RUN / 'object_statistics.csv')
img = pd.read_csv(RUN / 'image_statistics.csv')
class_table = pd.DataFrame({'objects': summary['objects_per_class'], 'share_pct': summary['class_share_pct'], 'images': summary['images_per_class']})
display(class_table)
print('Imbalance ratio:', summary['imbalance_ratio'])
display(pd.crosstab(obj['class_name'], obj['split'], normalize='columns').round(3))
display(Image(filename=str(RUN / 'figures' / 'objects_per_class.png')))
display(Image(filename=str(RUN / 'figures' / 'classes_by_split.png')))

## 4. EDA — obrazy i bounding boxy

Jasność to średnia skala szarości, kontrast to jej odchylenie standardowe, a ostrość to wariancja Laplacianu. Są to wskaźniki do inspekcji, nie etykiety jakości. Współrzędne ramek są znormalizowane do [0, 1].

In [ ]:
display(img.groupby('split')[['width','height','file_size_kb','brightness','contrast','sharpness','n_objects']].agg(['mean','median']).round(2))
display(obj.groupby('class_name')[['bw','bh','area','box_aspect_ratio']].median().round(3))
print('Rozdzielczości:', summary['unique_resolutions'])
print('Małe boxy <1% obrazu:', summary['small_objects_area_lt_1pct'])
print('Duże boxy >50% obrazu:', summary['large_objects_area_gt_50pct'])
print('Środki w środkowej połowie obrazu:', summary['central_boxes_center_in_middle_half'])
for name in ['image_width.png','image_height.png','image_aspect_ratio.png','image_file_size_kb.png','image_brightness.png','image_contrast.png','image_sharpness.png','brightness_by_class.png','contrast_by_class.png','sharpness_by_class.png','box_bw.png','box_bh.png','box_area.png','box_width_height.png','box_centers.png','box_area_by_class.png']:
    display(Image(filename=str(RUN / 'figures' / name)))

## 5. Przykłady z prawdziwymi adnotacjami

Zielone ramki oznaczają rzeczywiste obiekty albo ramki wyznaczone z ich oryginalnych poligonów.

In [ ]:
for name in ['most_objects.png','smallest_objects.png','largest_objects.png','dark_scenes.png','bright_scenes.png','blurry_scenes.png','challenging_scenes.png']:
    display(Image(filename=str(RUN / 'figures' / name)))
for class_name in summary['classes']:
    display(Image(filename=str(RUN / 'figures' / f"examples_{class_name.lower()}.png")))

## 6. Trening baseline i GreenAI

Pretrained YOLO11n, seed 42, standardowe ustawienia Ultralytics, 3 epoki zgodnie z decyzją projektową. CodeCarbon obejmuje wyłącznie okno treningowe z walidacją epok. Na tym komputerze PyTorch nie udostępnia MPS, więc trening jest CPU. Wejście 320 px ogranicza czas uruchomienia Nano i odpowiada mobilnemu punktowi odniesienia; przy dużym udziale małych obiektów może obniżyć ich wykrywalność.

**Uwaga o odtwarzaniu:** zapisany przebieg początkowo miał limit 10 epok i został przerwany po trzeciej na życzenie użytkownika. Komenda `--epochs 3` uruchamia nowy, podobny krótki eksperyment; harmonogram learning rate może być inny, więc wagi nie będą identyczne.

In [ ]:
# !python -m ml_baseline.audit --dataset 'Trash Detection Dataset/CUSTOM_DATASET' --out runs/academic_3_epoch_example
# Po przejrzeniu audytu uruchom osobny audyt dla nowego katalogu wyników, a potem trening w terminalu:
# python -m ml_baseline.train --dataset 'Trash Detection Dataset/CUSTOM_DATASET' --out runs/academic_3_epoch_example --epochs 3 --batch 32 --imgsz 320 --patience 5 > runs/academic_3_epoch_example/training.log 2>&1
config = yaml.safe_load((RUN / 'config.yaml').read_text()) if (RUN / 'config.yaml').exists() else {}
print(config)
if (RUN / 'training_metrics.csv').exists():
    display(pd.read_csv(RUN / 'training_metrics.csv').tail())

## 7. Ewaluacja `best.pt`

Skrypt ocenia najpierw validation, następnie test. Metryki per klasa pochodzą z Ultralytics; F1 jest średnią harmoniczną precision i recall. Natywne wykresy obejmują macierze pomyłek oraz krzywe PR i confidence.

In [ ]:
if (RUN / 'final_metrics.json').exists():
    final = json.loads((RUN / 'final_metrics.json').read_text())
    green = json.loads((RUN / 'green_ai.json').read_text())
    display(pd.DataFrame(final['per_class']).T)
    display(pd.Series({k: final.get(k) for k in ['precision','recall','f1_score','map50','map50_95','best_epoch','epochs_executed','training_time_seconds','model_size_mb']}))
    display(pd.Series(green))
    for name in ['test_eval_confusion_matrix.png','test_eval_confusion_matrix_normalized.png','test_eval_BoxPR_curve.png','test_eval_BoxF1_curve.png','test_eval_BoxP_curve.png','test_eval_BoxR_curve.png']:
        p = RUN / 'figures' / name
        if p.exists(): display(Image(filename=str(p)))
else: print('Trening i test jeszcze nie są zakończone.')

## 8. Podsumowanie i wnioski

Wnioski są zapisane po rzeczywistym treningu w `runs/baseline_yolo_nano/REPORT.md`. W raporcie należy rozróżniać wynik obserwowany na obecnym datasecie od oczekiwanej jakości zdjęć z telefonu; tego drugiego nie da się ocenić bez zdjęć z docelowej domeny.

In [ ]:
report = RUN / 'REPORT.md'
if report.exists(): display(Markdown(report.read_text()))
else: print('Raport końcowy będzie dostępny po ewaluacji.')